[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *Plotnine*.

*Plotnine* es un proyecto que pretende implementar en *Python* las funcionalidades de [*ggplot2*](https://ggplot2.tidyverse.org/), la popular herramienta de visualización de datos para *R* y la [gramática de gráficas por capas](http://vita.had.co.nz/papers/layered-grammar.html); ambas creadas por [*Hadley Wickham*](https://hadley.nz/) y basadas en la gramática de gráficas de [*Leland Wilkinson*](https://en.wikipedia.org/wiki/Leland_Wilkinson).

Aún cuando muchas de las funcionalidades de *ggplot2* ya han sido portadas, aún quedan muchas que se encuentran pendientes.


La documentación oficial de *Plotnine* puede ser consultada en la siguiente liga:

https://plotnine.readthedocs.io/en/stable/

La siguiente liga apunta a un breve tutorial sobre *Plotnine* tomando como base a *ggplot2*: 

https://datascienceworkshops.com/blog/plotnine-grammar-of-graphics-for-python/

In [ ]:
%pip install plotnine

In [ ]:
from plotnine import *
import pandas as pd
import numpy as np
from datetime import datetime
from typing import Any

## Gramática de capas de un gráfico.

La gramática de capas define una estructura de elementos que conforman un gráfico.

* Datos.
* Mapeo.
* Estética.
* Objetos geométricos.
* Escalas.
* Especificación de facetas.
* Transformaciones.
* Sistema de coordenadas.

## Sintaxis de la gramática.

### La función ```ggplot()```.

```
ggplot(data=<datos>, mapping=<estética>, <argumentos>)
```

https://plotnine.readthedocs.io/en/stable/generated/plotnine.ggplot.html

### La función ```plotnine.mapping.aes()```.

https://plotnine.readthedocs.io/en/stable/generated/plotnine.mapping.aes.html

### Funciones de geometría.

https://plotnine.readthedocs.io/en/stable/api.html#geoms

## Funciones de temas.

https://plotnine.readthedocs.io/en/stable/api.html#themes

## Ejemplos.

### Ejemplo de histograma.

In [ ]:
np.random.seed(23523889)

## Dataframe ilustrativo.

Los ejemplos de histograma y columnas usarán datos generados aleatoriamente con *NumPy*. Para garantizar reproducibilidad, la siguiente celda fijará la semilla del generador aleatorio.

In [ ]:
arreglo_base = pd.DataFrame(np.random.normal(12, 25, 1000),
                            columns=pd.Index(['observaciones']))
arreglo_base

* La siguiente celda creará el *dataframe* `arreglo_base` con 1000 observaciones de una distribución normal (media=12, desviación estándar=25), que servirá como datos de ejemplo para los gráficos de histograma.

In [ ]:
ggplot(data=arreglo_base)

* La siguiente celda creará un objeto `ggplot()` vacío a partir de `arreglo_base`. Sin capas geométricas, el resultado es un lienzo en blanco que muestra el sistema de coordenadas pero sin datos dibujados.

In [ ]:
ggplot(data=arreglo_base, mapping=aes(x='observaciones')) + geom_histogram()

* La siguiente celda agregará la geometría `geom_histogram()` al objeto `ggplot()` para construir el histograma completo con los parámetros por defecto.

In [ ]:
(ggplot(data=arreglo_base, mapping=aes(x='observaciones')) + 
geom_histogram(bins=10, fill='yellow', color="orange"))

* La siguiente celda personalizará el histograma ajustando el número de `bins` y los colores de relleno y borde con los parámetros `fill` y `color`.

In [ ]:
histograma = pd.DataFrame(np.histogram(arreglo_base, bins=13)).T
histograma.columns = pd.Index(['frecuencias','rangos'])
histograma

* La siguiente celda calculará las frecuencias del histograma con `np.histogram()` y construirá el *dataframe* `histograma` con columnas `frecuencias` y `rangos`, que se usará en el ejemplo de columnas.

### Ejemplo de columnas.

In [ ]:
ggplot(histograma, aes(x='rangos', y='frecuencias', fill='rangos')) + geom_col()

### Ejemplo de líneas con datos de ventas mensuales.

In [ ]:
np.random.seed(42)
fechas = pd.date_range('2020-01-01', periods=48, freq='ME')
ventas_mensuales = pd.DataFrame({
    'CDMX': np.random.normal(80_000, 15_000, 48).clip(20_000).astype(int),
    'MTY':  np.random.normal(55_000, 10_000, 48).clip(10_000).astype(int),
    'GDL':  np.random.normal(45_000,  8_000, 48).clip(10_000).astype(int),
}, index=fechas)
ventas_mensuales['Nacional'] = ventas_mensuales.sum(axis=1)
ventas_mensuales.index.name = 'fecha'
ventas_mensuales

* La siguiente celda generará el *dataframe* `ventas_mensuales` con ventas ficticias mensuales para las sucursales `CDMX`, `MTY` y `GDL` durante 48 meses (2020–2023), más el total `Nacional`.

In [ ]:
(ggplot(ventas_mensuales, aes(x=ventas_mensuales.index, y='Nacional'))
+ geom_line())

In [ ]:
(ggplot(ventas_mensuales, aes(x=ventas_mensuales.index, y='Nacional'))
+ geom_line()
+ geom_smooth())

In [ ]:
(ggplot(ventas_mensuales, aes(x=ventas_mensuales.index, y='Nacional'))
+ geom_line()
+ geom_smooth(span=0.3, color='red'))

In [ ]:
(ggplot(ventas_mensuales, aes(x=ventas_mensuales.index, y='Nacional'))
 + geom_line()
 + geom_smooth(span=0.3, color='blue')
 + theme_xkcd()
 + theme(axis_text_x=element_text(rotation=90, hjust=0.5)))

In [ ]:
(ggplot(ventas_mensuales, aes(x=ventas_mensuales.index, y='Nacional'))
 + geom_line(color='red')
 + geom_smooth(span=0.3, color='blue')
 + theme_tufte()
 + theme(axis_text_x=element_text(rotation=45, hjust=1)))

### Comparación de ventas por sucursal en un mes.

In [ ]:
data = ventas_mensuales.drop('Nacional', axis=1).loc['2021-01-31'].to_frame()
data.columns = pd.Index(['Ventas'])
data.index.name = 'Sucursal'
data

* La siguiente celda seleccionará las ventas del mes de enero de 2021 por sucursal para construir el *dataframe* `data`, que se utilizará como base del gráfico de columnas.

In [ ]:
(ggplot(data, aes(x=data.index, y='Ventas', fill='Ventas'))
 + geom_col()
 + theme(axis_text_x=element_text(rotation=45, hjust=1)))

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>